In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Sayantan Nandi\\Desktop\\mlops\\datascienceproject\\research'

In [3]:
os.chdir("../")

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: str
    unzip_dir: Path
    
    

In [5]:
from src.datascience.constants import *
from src.datascience.utils.common import *

In [10]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH, schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_url=config.source_URL,local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config

In [7]:
import os
import urllib.request as request
from src.datascience import logger
import zipfile


In [12]:
class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config
        
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_url,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info(f"File already exists")
            
    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [13]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-30 00:58:43,072: INFO: common: Directory created or already exists: artifacts]
[2026-03-30 00:58:43,074: INFO: common: Directory created or already exists: artifacts/data_ingestion]
[2026-03-30 00:58:43,867: INFO: 2348010268: artifacts/data_ingestion/data.zip downloaded! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: AFF2:224A16:606212:AA13D2:69C97D78
Accept-Ranges: bytes
Date: Sun, 29 Mar 2026 19:28:58 GMT
Via: 1.1 varnish
X-Served-By: cache-bom-vanm7210084-BOM
X-Cache: MISS
X-Cache-Hits: 0
X-Timer: S1774812538.008477,VS0,VE263
Vary: Authorization,Accept-Encoding
Access-Control-Allow-Origin: *
Cross-Origin-